# Ch1 Autograd 教案 — 打开深度学习的引擎盖

---

## 课程信息

| 项目 | 内容 |
|:---|:---|
| **课程标题** | Ch1: Autograd 可视化 — 打开深度学习的引擎盖 |
| **预计时长** | ~90 分钟 |
| **源文件** | `Ch1_Autograd/Ch1_Autograd.ipynb` |
| **核心问题** | 梯度是如何自动计算出来的？`loss.backward()` 背后到底发生了什么？ |

---

## 时间总表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00–05:00 | 开场 + 前置知识回顾 | Cell 0–2 | 5 min |
| 05:00–10:00 | 环境准备 + 计算图直觉图 | Cell 3–7 | 5 min |
| 10:00–25:00 | 手写 Micrograd：Value 类实现 | Cell 6–8 | 15 min |
| 25:00–35:00 | 深入理论：自动微分数学基础 + 测试 Micrograd | Cell 9–11 | 10 min |
| 35:00–40:00 | **休息 + 回顾** | — | 5 min |
| 40:00–50:00 | 可视化计算图（DAG 绘制） | Cell 12–13 | 10 min |
| 50:00–60:00 | 用 Micrograd 训练神经网络 | Cell 14–16 | 10 min |
| 60:00–67:00 | 训练循环理论 + Loss 曲线可视化 | Cell 17–18 | 7 min |
| 67:00–75:00 | 对接 PyTorch 验证 | Cell 19–20 | 8 min |
| 75:00–80:00 | **休息 + 回顾** | — | 5 min |
| 80:00–85:00 | 链式法则深入 + 可视化 | Cell 21–23 | 5 min |
| 85:00–95:00 | 动手练习：实现 exp 和 relu | Cell 24–25 | 10 min |
| 95:00–100:00 | 总结 + 下一章预告 | Cell 26–28 | 5 min |

---

## 课前准备清单

- [ ] 确认 Python 环境可用，`numpy`、`matplotlib`、`torch` 已安装
- [ ] 打开源 notebook `Ch1_Autograd/Ch1_Autograd.ipynb`，Kernel 选择正确
- [ ] 预跑一遍所有 Cell，确认无报错（特别是字体配置 Cell 5、7）
- [ ] 准备白板/画板用于手画计算图
- [ ] 确认投影/屏幕共享正常
- [ ] 准备备用字体路径（如果 `NotoSansCJKsc-Regular.otf` 不存在）


## 段落 0：开场与前置知识回顾

---

📍 **运行 Cell 0–2**（标题页 + 学习路线 + 前置知识 Markdown）

⏱ **时间分配：5 分钟**

🎯 **本段目标**
- 建立课程全局视角：本章在整个课程中的位置
- 快速回顾前置概念（权重、损失函数、梯度）
- 激发好奇心：`loss.backward()` 到底干了什么？

🗣 **讲课话术**

> 大家好，欢迎来到第一章。上一章 Ch0 我们跑通了一个完整的训练循环——前向传播、算 Loss、反向传播、更新参数——四步走。
>
> 但是当时我们调用 `loss.backward()` 的时候，是不是就像按了一个魔法按钮？Loss 一下子就把每个参数的梯度都算好了。今天我们就来**打开这个黑盒**，看看里面到底是怎么回事。
>
> 先快速回顾几个概念。损失函数是什么？它就是一个「打分器」，告诉我们模型有多「错」。梯度呢？梯度就是一个「方向标」，告诉我们每个参数应该往哪个方向调、调多少，才能让 Loss 变小。
>
> 那问题来了：一个大模型可能有几百万、甚至上千亿个参数，PyTorch 怎么**一次性**把所有参数的梯度都算出来的？答案就是**自动微分（Autograd）**。今天我们不光要理解它，还要亲手写一个！

👀 **输出要点**
- Cell 0–2 均为 Markdown，无代码输出
- 重点让学生看到学习路线表和前置知识的公式：$参数_{new} = 参数_{old} - 学习率 \times 梯度$

❓ **预判问题**

**Q: 梯度和导数有什么区别？**
> A: 对于单变量函数，梯度就是导数。对于多变量函数，梯度是所有偏导数组成的向量。在深度学习中我们通常说「梯度」，因为参数是多维的。

**Q: 为什么不直接用数学公式手算梯度？**
> A: 好问题！对于简单函数当然可以。但神经网络有几百万个参数、几百层运算，手算不现实。这正是自动微分要解决的问题，后面我们会详细对比不同的微分策略。

➡️ **转场**

> 好，概念回顾完毕。我们先把环境准备好，然后就开始动手写代码。


## 段落 1：环境准备 + 计算图直觉图

---

📍 **运行 Cell 3–7**（环境准备 + 可视化前向/反向传播图）

⏱ **时间分配：5 分钟**

🎯 **本段目标**
- 完成环境配置（numpy、matplotlib、字体）
- 用一张直觉图建立对「前向传播 + 反向传播」的整体印象
- 让学生记住关键数值：`L = (a*b+c)*d` 中 `a=2, b=3, c=4, d=5 → L=50`

🗣 **讲课话术**

> 先跑 Cell 4 和 Cell 5 把环境准备好。Cell 4 是 pip install，如果已经装好了可以跳过。Cell 5 设置了中文字体，这样后面画图时中文不会变成方块。
>
> 好，现在看 Cell 7 这张图。这是今天最重要的一张图，大家仔细看。上半部分是**前向传播**——数值从左往右流。`a=2, b=3`，乘一下得到 `ab=6`，加上 `c=4` 得到 `s=10`，再乘 `d=5` 得到最终结果 `L=50`。
>
> 下半部分就是**反向传播**——梯度从右往左流回去。`L` 的梯度是 1（对自己求导嘛），然后通过乘法节点，`s` 的梯度是 `d=5`，`d` 的梯度是 `s=10`。再往回走，`ab` 的梯度是 5，`c` 的梯度也是 5。最后 `a` 的梯度是 `5×3=15`，`b` 的梯度是 `5×2=10`。
>
> 注意每条边上标的是**局部导数**。比如 `dL/ds = d = 5`，这是乘法的局部导数——对 `s` 求导就是另一个因子 `d`。这就是链式法则：**上游梯度 × 局部导数 = 当前梯度**。

👀 **输出要点**
- Cell 7 输出一张双层计算图：上层为前向数值（蓝/黄/紫节点），下层为反向梯度（黄色节点）
- 关键数值：`a` 的梯度 = 15，`b` 的梯度 = 10，`c` 的梯度 = 5，`d` 的梯度 = 10
- 底部公式：`dL/da = (dL/ds)*(ds/d(ab))*(d(ab)/da) = 5*1*3 = 15`

❓ **预判问题**

**Q: 为什么 L 的梯度是 1？**
> A: 因为 $\frac{\partial L}{\partial L} = 1$，任何变量对自己的导数都是 1。这是反向传播的起点。

**Q: 图里字体显示不出来怎么办？**
> A: 检查 Cell 7 中的字体路径 `../assets/fonts/NotoSansCJKsc-Regular.otf` 是否存在。如果不存在，可以用 Cell 5 中设置的 `Microsoft YaHei` 或 `SimHei` 替代。

➡️ **转场**

> 这张图给了我们直觉，但代码怎么实现呢？接下来我们就亲手写一个自动微分引擎。


## 段落 2：手写 Micrograd — Value 类核心实现

---

📍 **运行 Cell 6（Markdown 说明）+ Cell 8（Value 类代码）**

⏱ **时间分配：15 分钟**

🎯 **本段目标**
- 理解 `Value` 类的四个核心属性：`data`、`grad`、`_backward`、`_prev`
- 理解加法和乘法的 `_backward` 实现（局部梯度公式）
- 理解 `backward()` 方法中的拓扑排序 + 反向遍历

🗣 **讲课话术**

> 这是今天最核心的代码——50 行左右的 `Value` 类。大家跟我一起读。
>
> 首先看 `__init__`。每个 `Value` 对象有四样东西：
> - `data`：数值本身，比如 2.0
> - `grad`：梯度，初始化为 0
> - `_backward`：一个函数，告诉这个节点怎么把梯度传给它的「父节点」
> - `_prev`：产生这个值的子节点集合
>
> 然后看 `__add__`。当我们写 `a + b` 时，Python 会调用这个方法。它做两件事：
> 1. **前向**：算出 `out.data = self.data + other.data`
> 2. **记录反向规则**：加法的梯度直接传递，所以 `_backward` 里写的是 `self.grad += out.grad` 和 `other.grad += out.grad`
>
> 为什么是 `+=` 而不是 `=`？这非常关键！因为一个变量可能被用在多个地方。比如 `a + a`，`a` 被用了两次，两条路径的梯度要**累加**。
>
> 再看 `__mul__`。乘法的梯度是「交叉相乘」——对 `a` 求导得到 `b`，对 `b` 求导得到 `a`。所以 `self.grad += other.data * out.grad`。
>
> 最后看 `backward()`。它做两步：
> 1. **拓扑排序**：用 DFS 遍历计算图，确保每个节点在它的依赖之后被处理
> 2. **反向传播**：从输出开始（`self.grad = 1.0`），按逆拓扑序依次调用每个节点的 `_backward()`
>
> 运行一下——输出 "Value 类定义完成！"。代码不长，但这就是 PyTorch `autograd` 的核心思想。

👀 **输出要点**
- Cell 8 输出：`Value 类定义完成！`
- Value 类共支持 5 种运算：`+`、`*`、`**`、`tanh`、`-`（通过 `__neg__` 和 `__sub__`）

❓ **预判问题**

**Q: `_backward` 为什么用闭包（closure）而不是类方法？**
> A: 因为每个运算实例的 `_backward` 逻辑不同——它需要捕获 `self`、`other`、`out` 这些具体的 Value 对象。闭包天然做到了这一点。

**Q: 拓扑排序为什么是必要的？**
> A: 如果不做拓扑排序，可能在某个节点的上游梯度还没算好之前就调用了它的 `_backward`，导致梯度不完整。拓扑排序保证了正确的计算顺序。

**Q: `__radd__` 和 `__rmul__` 是干什么的？**
> A: 当左操作数不是 Value 时（比如 `2 + a`），Python 会调用右操作数的 `__radd__`。这样 `2 + a` 和 `a + 2` 都能正常工作。

➡️ **转场**

> Value 类写好了。但它真的能算对梯度吗？接下来我们先看看理论基础，然后实际测试一下。


## 段落 3：自动微分理论 + 测试 Micrograd

---

📍 **运行 Cell 9（理论 Markdown）+ Cell 10–11（测试代码）**

⏱ **时间分配：10 分钟**

🎯 **本段目标**
- 理解三种微分策略的对比：数值微分 vs 符号微分 vs 自动微分
- 理解为什么深度学习选择**反向模式 AD**
- 验证 Micrograd 的梯度计算正确

🗣 **讲课话术**

> 在测试之前，我们先花几分钟理解一下为什么自动微分这么重要。Cell 9 里对比了三种计算梯度的方法。
>
> **数值微分**：最笨的办法，把每个参数加一点点扰动，看输出变了多少。如果有 n 个参数，就要跑 n 次前向传播。GPT-3 有 1750 亿参数，那就要跑 3500 亿次——完全不可能。
>
> **符号微分**：像 Mathematica 那样做符号求导。问题是表达式会爆炸性膨胀——几百层网络求导后的公式能写满一本书。
>
> **自动微分**：我们刚写的 Micrograd 就是自动微分！它的优点是：不做符号化简（不膨胀），不做数值近似（没误差），而且反向模式只需要 **1 次前向 + 1 次反向**，不管有多少参数！
>
> 好，现在来验证。运行 Cell 11：`a=2.0, b=-3.0, c=10.0`，计算 `d = a*b + c = 2×(-3)+10 = 4.0`。反向传播后：
> - `a.grad = -3.0`（∂d/∂a = b = -3）
> - `b.grad = 2.0`（∂d/∂b = a = 2）
> - `c.grad = 1.0`（∂d/∂c = 1，加法梯度直传）
>
> 全部正确！和手算完全一致。

👀 **输出要点**（Cell 11 输出）
```
前向传播: d = a * b + c = 2.0 * -3.0 + 10.0 = 4.0

反向传播后的梯度:
  a.grad = -3.0  (∂d/∂a = b = -3.0)
  b.grad = 2.0   (∂d/∂b = a = 2.0)
  c.grad = 1.0   (∂d/∂c = 1)
```

❓ **预判问题**

**Q: Cell 9 里那个手算例子 f(x,y)=x²y+xy² 太复杂了，需要全部讲吗？**
> A: 可以简要提一下关键点：y 被两个节点使用（fan-out），所以梯度需要累加两条路径的贡献（4+12=16）。这正是 `+=` 的实际意义。

**Q: 反向模式 AD 和前向模式 AD 到底差在哪？**
> A: 前向模式：固定一个输入，算它对所有输出的影响，n 个输入要跑 n 次。反向模式：固定一个输出（Loss），算所有输入对它的影响，1 个输出只跑 1 次。深度学习输出是 1 个标量 Loss，所以反向模式完胜。

➡️ **转场**

> 数值验证通过了，但我们能不能更直观地「看到」计算图长什么样？下面就来画一画。


## ☕ 休息（5 分钟）

---

### 前半段回顾（3 句话）

1. **自动微分的核心**：每次运算时记录「谁产生了谁」，构建一张计算图（DAG），反向传播时沿图逆序用链式法则算梯度。
2. **Value 类**：50 行代码实现了加法、乘法、幂函数、tanh 四种运算的前向计算和反向梯度传播，核心是 `_backward` 闭包和 `grad +=`。
3. **验证通过**：`d = a*b + c`，`a=2, b=-3, c=10` → `d=4`，梯度 `a.grad=-3, b.grad=2, c.grad=1`，全部正确。

### 下半段预告

接下来我们要：
- 把计算图**画出来**，直观看到节点和边
- 用 Micrograd **训练一个神经网络**（41 个参数的 MLP）
- 和 PyTorch 对比，验证我们的实现和 PyTorch 完全一致


## 段落 4：可视化计算图（DAG 绘制）

---

📍 **运行 Cell 12（Markdown）+ Cell 13（draw_dot + 神经元计算图）**

⏱ **时间分配：10 分钟**

🎯 **本段目标**
- 理解 `trace()` 函数如何遍历计算图收集节点和边
- 直观看到一个神经元的完整计算图
- 将 DAG 中的节点与 Value 类的属性对应起来

🗣 **讲课话术**

> 现在来画计算图。Cell 13 的代码比较长，但核心就两个函数：
>
> `trace(root)`：从输出节点开始，递归遍历 `_prev`，收集所有节点和边。这就是在遍历我们之前构建的 DAG。
>
> `draw_dot(root)`：用 matplotlib 把这个 DAG 画出来。蓝色方框是数值节点，黄色圆圈是运算节点。
>
> 这里我们构建了一个**单神经元**的计算图：两个输入 `x1=2.0, x2=0.0`，两个权重 `w1=-3.0, w2=1.0`，一个偏置 `b=6.8814`。计算过程是 `n = x1*w1 + x2*w2 + b`，然后过 `tanh` 激活得到输出 `o`。
>
> 看输出：`o = tanh(0.8814) = 0.7071`。各参数的梯度：
> - `w1.grad = 1.0000`——这个梯度告诉我们 w1 增大 1，输出 o 也增大约 1（在当前工作点附近）。
> - `w2.grad = 0.0000`——因为 x2=0，所以 w2 对输出没有影响。
> - `b.grad = 0.5000`——偏置的梯度是 0.5，说明 tanh 在这个点的导数是 0.5。
>
> 注意看图里每个节点都显示了 `data` 和 `grad` 两个值——上面是前向计算的数值，下面是反向传播的梯度。

👀 **输出要点**（Cell 13 输出）
```
计算图已构建，梯度已计算！

输出: o = tanh(0.8814) = 0.7071

各参数梯度:
  w1.grad = 1.0000
  w2.grad = 0.0000
  b.grad = 0.5000

计算图已保存为 computation_graph.svg
```
- 还会显示一张 matplotlib 绘制的计算图，节点从左到右排列

❓ **预判问题**

**Q: 为什么 w2 的梯度是 0？**
> A: 因为 `x2=0.0`，所以 `x2*w2=0`，无论 w2 怎么变，这一项都是 0，对输出没影响，梯度自然是 0。如果把 x2 改成非零值，w2 的梯度就不为 0 了。

**Q: tanh(0.8814) 怎么刚好等于 0.7071？**
> A: 偏置 `b=6.8813735870195432` 是精心选择的，使得 `n = -6+0+6.8814 = 0.8814`，`tanh(0.8814) ≈ 0.7071 ≈ √2/2`。这是为了让数值好看，方便教学。

➡️ **转场**

> 好，计算图画出来了，一个神经元的结构一目了然。但一个神经元太简单了，让我们搭一个真正的神经网络，用我们自己写的引擎来训练它！


## 段落 5：用 Micrograd 训练神经网络

---

📍 **运行 Cell 14（Markdown）+ Cell 15（Neuron/Layer/MLP 类）+ Cell 16（训练循环）**

⏱ **时间分配：10 分钟**

🎯 **本段目标**
- 理解从 Value → Neuron → Layer → MLP 的抽象层次
- 掌握训练循环的四步：前向 → Loss → 反向 → 更新
- 观察 Loss 从大到小的下降过程

🗣 **讲课话术**

> 现在我们在 Value 类的基础上搭建神经网络。Cell 15 定义了三个类：
>
> **Neuron**：一个神经元，有 n 个权重和 1 个偏置。`__call__` 做的就是 `sum(w*x) + b`，然后过 `tanh`。
>
> **Layer**：一层里有多个 Neuron，每个 Neuron 独立计算。
>
> **MLP**：多层感知机，把多个 Layer 串起来。
>
> 我们创建了一个 `MLP(3, [4, 4, 1])` —— 3 个输入，两个隐藏层各 4 个神经元，1 个输出。一共**41 个参数**。
>
> 然后 Cell 16 做训练。训练数据只有 4 个样本，目标值是 `[1, -1, -1, 1]`。看训练过程：
> - Epoch 20：Loss = 0.060669，已经很小了
> - Epoch 40：Loss = 0.021675
> - Epoch 100：Loss = 0.007151
>
> 最终预测 `[0.97, -0.97, -0.95, 0.95]`，和目标 `[1, -1, -1, 1]` 非常接近！
>
> 100 轮训练只用了 **0.12 秒**。当然这是因为只有 41 个参数。PyTorch 训练几亿参数的模型用的也是同样的原理，只是用了 GPU 加速和更高效的张量运算。
>
> 注意训练循环里有一行 `p.grad = 0.0`——每轮开始前**必须清零梯度**。为什么？因为 `backward()` 里梯度是累加的（`+=`）。如果不清零，上一轮的梯度会和这一轮的叠加，更新方向就错了。这对应 PyTorch 里的 `optimizer.zero_grad()`。

👀 **输出要点**（Cell 15 + Cell 16 输出）
```
模型参数量: 41

开始训练...
==================================================
Epoch  20 | Loss: 0.060669
Epoch  40 | Loss: 0.021675
Epoch  60 | Loss: 0.012996
Epoch  80 | Loss: 0.009238
Epoch 100 | Loss: 0.007151
==================================================
训练完成! 耗时: 0.12 秒

最终预测: [0.97, -0.97, -0.95, 0.95]
目标值:    [1.0, -1.0, -1.0, 1.0]
```

❓ **预判问题**

**Q: 为什么学习率是 0.05？换个值会怎样？**
> A: 0.05 是经验值。太大（如 1.0）Loss 会震荡甚至爆炸；太小（如 0.0001）收敛极慢。Cell 17 的理论部分有详细解释。可以让学生课后尝试修改学习率看效果。

**Q: 为什么 Loss 用的是 MSE（均方误差）而不是其他？**
> A: 这是回归任务最常用的 Loss。`(yout - ygt)**2` 就是 MSE 的每个样本分量。交叉熵更适合分类任务。

**Q: 41 个参数怎么算出来的？**
> A: 第一层：4 个 Neuron，每个 3 个权重 + 1 个偏置 = 4×4 = 16。第二层：4 个 Neuron，每个 4 个权重 + 1 个偏置 = 4×5 = 20。输出层：1 个 Neuron，4 个权重 + 1 个偏置 = 5。总计 16+20+5 = 41。

➡️ **转场**

> 训练成功了！但这些数字的变化趋势更直观的方式是画条曲线。另外，理论部分还有一些重要知识点——梯度清零、三种梯度下降、学习率的影响——我们快速过一下。


## 段落 6：训练循环理论 + Loss 曲线可视化

---

📍 **运行 Cell 17（理论 Markdown）+ Cell 18（Loss 曲线图）**

⏱ **时间分配：7 分钟**

🎯 **本段目标**
- 理解为什么必须清零梯度（`p.grad = 0.0` / `optimizer.zero_grad()`）
- 了解 BGD vs SGD vs Mini-batch SGD 的区别
- 直观看到 Loss 下降曲线

🗣 **讲课话术**

> Cell 17 里有三个重要知识点，我挑最关键的讲。
>
> 第一，**梯度清零**。我们的 `backward()` 里用的是 `grad +=`，所以如果不手动清零，上一轮的梯度会一直叠加。PyTorch 也是一样的设计——每轮训练开头必须调 `optimizer.zero_grad()`。不过有趣的是，这个「累加」设计在某些场景下反而有用：当 GPU 显存不够时，可以用多个小 batch 分别算梯度，累加起来模拟一个大 batch。
>
> 第二，**三种梯度下降**。实际中用得最多的是 Mini-batch SGD：每次用一小批样本（32-512 个）算梯度。它在计算效率和优化效果之间取了平衡。我们的例子只有 4 个样本，所以等效于 BGD（批量梯度下降）。
>
> 第三，**学习率的影响**。Cell 17 里有个很好的类比——蒙着眼走山路：步子太大跨到对面山坡（震荡），步子太小天黑了还走不到谷底（收敛慢）。
>
> 好，运行 Cell 18 看 Loss 曲线。这条蓝色曲线从 Epoch 0 的高点快速下降，到 Epoch 20 左右基本收敛，后面缓慢下降。这就是典型的训练曲线形状——先快后慢。

👀 **输出要点**
- Cell 18 输出一张 Loss 曲线图，横轴 Epoch（0-100），纵轴 Loss
- 曲线从高点快速下降到 ~0.06（Epoch 20），然后缓慢降至 ~0.007（Epoch 100）

❓ **预判问题**

**Q: 梯度累加和 Batch Normalization 有什么关系？**
> A: 两者是独立的概念。梯度累加是优化技巧，BN 是网络结构。但梯度累加时要注意 BN 的统计量（mean/var）只在当前 mini-batch 上计算，和大 batch 的统计量可能不同。

➡️ **转场**

> Loss 曲线看着没问题。但最终极的验证是——我们的 Micrograd 和 PyTorch 算的梯度一不一样？


## 段落 7：对接 PyTorch 验证

---

📍 **运行 Cell 19（Markdown）+ Cell 20（PyTorch 对比代码）**

⏱ **时间分配：8 分钟**

🎯 **本段目标**
- 用 PyTorch 重新计算同一个神经元的梯度
- 验证 Micrograd 和 PyTorch 结果完全一致
- 建立信心：我们手写的引擎原理正确

🗣 **讲课话术**

> 现在到了激动人心的验证环节。Cell 20 用 PyTorch 重新算了我们之前那个神经元的梯度。
>
> 注意 PyTorch 的写法：`torch.tensor(2.0, requires_grad=True)` 就相当于我们的 `Value(2.0)`。`requires_grad=True` 告诉 PyTorch「我要跟踪这个张量的梯度」。
>
> 然后 `o_pt.backward()` 就相当于我们的 `o.backward()`。
>
> 看结果：
>
> | 参数 | PyTorch | Micrograd |
> |:---|:---|:---|
> | w1.grad | 1.0000 | 1.0000 |
> | w2.grad | 0.0000 | 0.0000 |
> | b.grad | 0.5000 | 0.5000 |
>
> **完全一致！** 我们用 50 行代码实现的 Micrograd，和 PyTorch 的 autograd 引擎算出了一模一样的梯度。
>
> 当然，PyTorch 的实现要复杂得多——它支持 GPU、支持张量运算、支持几百种操作——但**核心原理**和我们写的是一样的：构建计算图，反向传播链式法则。

👀 **输出要点**（Cell 20 输出）
```
PyTorch 计算的梯度:
  w1.grad = 1.0000
  w2.grad = 0.0000
  b.grad = 0.5000

我们 Micrograd 计算的梯度:
  w1.grad = 1.0000
  w2.grad = 0.0000
  b.grad = 0.5000

✓ 两者完全一致！我们的实现是正确的！
```

❓ **预判问题**

**Q: PyTorch 的 `requires_grad` 不设为 True 会怎样？**
> A: 那 PyTorch 就不会为这个张量建计算图，`backward()` 后 `.grad` 是 `None`。这可以节省内存。推理时（不需要训练）就应该关闭它，或者用 `torch.no_grad()` 上下文。

**Q: Micrograd 和 PyTorch 的实际区别是什么？**
> A: Micrograd 只支持标量运算（一个 Value 存一个数），PyTorch 支持张量运算（一个 Tensor 存一整个矩阵/向量），可以利用 GPU 并行加速。另外 PyTorch 支持几百种操作（卷积、注意力等），而我们只实现了 5 种。

➡️ **转场**

> 验证通过！在进入练习之前，我们再深入理解一下链式法则的数学和直觉。


## ☕ 休息（5 分钟）

---

### 后半段回顾（3 句话）

1. **计算图可视化**：用 `trace()` 遍历 DAG，`draw_dot()` 画出节点和边。单神经元的计算图：`o = tanh(x1*w1 + x2*w2 + b) = 0.7071`。
2. **MLP 训练成功**：41 个参数的小网络，100 轮训练 Loss 从高点降到 0.007，预测值 `[0.97, -0.97, -0.95, 0.95]` 接近目标 `[1, -1, -1, 1]`。
3. **PyTorch 验证通过**：`w1.grad=1.0, w2.grad=0.0, b.grad=0.5`，Micrograd 和 PyTorch 结果完全一致。

### 接下来

- 链式法则的水流类比和数值示例
- 动手练习：为 Value 类添加 `exp` 和 `relu`


## 段落 8：链式法则深入 + 可视化

---

📍 **运行 Cell 21–22（Markdown 理论）+ Cell 23（水流类比图）**

⏱ **时间分配：5 分钟**

🎯 **本段目标**
- 用水流类比直观理解链式法则
- 理解 VJP（向量-雅可比积）的概念
- 记住核心公式：「上游梯度 × 局部梯度 = 当前梯度」

🗣 **讲课话术**

> Cell 23 画了两张图。左边是**水流类比**：想象梯度像水一样流过管道，每经过一个函数节点，流量就乘以那个函数的导数。最终从输出流到输入的总量，就是链式法则的结果。
>
> 右边是一个具体例子：`y = (2x)²`，在 `x=3` 处求 `dy/dx`。
> - 前向：`u = 2x = 6`，`y = u² = 36`
> - 反向：`dy/du = 2u = 12`，`du/dx = 2`
> - 链式法则：`dy/dx = 12 × 2 = 24`
> - 验证：`y = 4x²`，`dy/dx = 8x = 24` ✓
>
> Cell 22 里有一个非常重要的公式框，大家记住这个：
>
> $$\frac{\partial L}{\partial u_i} \mathrel{+}= \frac{\partial L}{\partial v} \times \frac{\partial v}{\partial u_i}$$
>
> 翻译成人话就是：**当前节点的梯度 = 上游梯度 × 局部梯度**，如果有多条路径就**累加**。这一个公式就是反向传播的全部！

👀 **输出要点**
- Cell 23 输出两张并排图：左边「水流类比」，右边「具体数值 y=(2x)²」
- 左图关键文字："最终流量 = 1 × f'(x) × g'(f(x)) = 链式法则！"
- 右图验证：dy/dx = 12 × 2 = 24 = 8×3 ✓

❓ **预判问题**

**Q: VJP 是什么？前面提到了但没细讲。**
> A: VJP = Vector-Jacobian Product，就是「一个行向量乘以雅可比矩阵」。反向传播每一步做的就是 VJP。PyTorch 不会显式构造整个雅可比矩阵（太大了），而是直接算 VJP。对于标量操作（我们的 Micrograd），VJP 退化为简单的乘法。

➡️ **转场**

> 理论够了，现在是动手时间！让我们来扩展 Micrograd，给它添加两个新操作。


## 段落 9：动手练习 — 实现 exp 和 relu

---

📍 **运行 Cell 24（思考题 Markdown）+ Cell 25（练习代码）**

⏱ **时间分配：10 分钟**

🎯 **本段目标**
- 学生独立实现 `exp()` 和 `relu()` 方法
- 理解不同激活函数的局部导数
- 用数值微分验证自己的实现

🗣 **讲课话术**

> 好，现在是你们的时间！Cell 25 里有一个 `ValueV2` 类继承了 `Value`，需要你们实现两个方法：`exp()` 和 `relu()`。
>
> 提示就在 Cell 22 的那张局部导数表里：
> - `exp` 的导数：`d(e^x)/dx = e^x`，也就是说 exp 的导数还是自己！
> - `relu` 的导数：`x > 0` 时为 1，`x ≤ 0` 时为 0。就像一个开关——正数全放过，负数全挡住。
>
> 给你们 2 分钟先自己试试。

### 提示节奏

**0–2 分钟**：让学生自己尝试，不给提示

**2 分钟时**：第一个提示
> 提示 1：模式和 `tanh` 方法一样。先算前向值 `out = ValueV2(...)`，然后定义 `_backward` 闭包，在里面写 `self.grad += 局部导数 * out.grad`。

**4 分钟时**：给出关键代码
> 提示 2：
> - exp 的 `_backward`：`self.grad += np.exp(x) * out.grad`
> - relu 的 `_backward`：`self.grad += (1.0 if self.data > 0 else 0.0) * out.grad`

### 常见错误

| 错误 | 现象 | 修正 |
|:---|:---|:---|
| 用 `=` 而不是 `+=` | 多路径梯度丢失 | 改为 `self.grad +=` |
| relu 前向用 `abs(x)` | 负数变正数，不是 relu | 改为 `max(0, x)` |
| exp 忘记乘 `out.grad` | 链式法则断裂，梯度不对 | 加上 `* out.grad` |
| relu 边界条件写成 `x >= 0` 时为 1 | 通常影响不大，但严格说 `x=0` 时 relu 导数未定义 | 约定 `x=0` 时为 0 或 1 均可 |

### 验证标准

运行 Cell 25 后应看到：
```
=== 验证 exp ===
  exp(2) = 7.3891  (期望: 7.3891)
  d(exp(x))/dx at x=2 = 7.3891  (期望: 7.3891)
  数值微分验证: 7.3891
  误差: 0.00003695

=== 验证 relu ===
  relu(3) = 3.0000  (期望: 3.0)
  d(relu)/dx at x=3 = 1.0000  (期望: 1.0)
  relu(-2) = 0.0000  (期望: 0.0)
  d(relu)/dx at x=-2 = 0.0000  (期望: 0.0)

如果所有期望值匹配，恭喜你的实现正确！
```

关键检查点：
- exp(2) 的值和梯度**都是 7.3891**（exp 的独特性质：导数等于自身）
- relu(3) 梯度为 1，relu(-2) 梯度为 0
- 数值微分误差在 1e-4 量级（浮点精度范围内）

❓ **预判问题**

**Q: 数值微分的 h=1e-5 是怎么选的？**
> A: 太大则截断误差大（泰勒展开的高阶项），太小则浮点舍入误差大。1e-5 到 1e-7 通常是比较好的范围。

**Q: relu 在 x=0 处不可导怎么办？**
> A: 实践中约定为 0 即可。PyTorch 也是这么做的。训练时恰好 x=0 的概率极低，不影响收敛。

➡️ **转场**

> 做得好！你们现在不仅理解了自动微分的原理，还能自己扩展引擎了。最后我们来总结一下今天学到的所有内容。


## 段落 10：总结 + 下一章预告

---

📍 **运行 Cell 26–27（总结 + 下一步 Markdown）**

⏱ **时间分配：5 分钟**

🎯 **本段目标**
- 串联全章知识点
- 强调核心公式和面试考点
- 预告 Ch2 Embedding

🗣 **讲课话术**

> 我们来总结一下今天学到了什么。
>
> **第一**，自动微分的核心就三步：构建计算图（前向传播时自动完成）→ 拓扑排序 → 逆序链式法则。没有魔法，就是链式求导。
>
> **第二**，我们亲手写了 50 行的 `Value` 类，实现了加、乘、幂、tanh 四种操作。在此基础上搭了一个 41 参数的 MLP，训练 100 轮 Loss 从几降到 0.007。
>
> **第三**，我们和 PyTorch 对比验证——梯度完全一致。`loss.backward()` 背后没有黑魔法，PyTorch 只是把我们做的事情自动化、GPU 化了。
>
> Cell 26 里有一张核心公式速查表，建议大家截图保存。特别是这个：
>
> $$\frac{\partial L}{\partial u_i} \mathrel{+}= \frac{\partial L}{\partial v} \cdot \frac{\partial v}{\partial u_i}$$
>
> 上游梯度乘以局部梯度，多路径累加。记住这一条就够了。
>
> Cell 26 最后还有三道面试题，大家课后可以自己练练。
>
> 下一章 Ch2 我们进入 **Embedding** 的世界——机器怎么把文字变成数字？词向量是怎么训练出来的？王和皇后之间的数学关系是什么？非常有趣，敬请期待。

👀 **输出要点**
- Cell 26–27 均为 Markdown，无代码输出
- 关键内容：核心概念图谱、公式速查表、三道面试题、延伸阅读链接

❓ **预判问题**

**Q: 课后应该怎么复习？**
> A: 三件事：(1) 从头到尾自己跑一遍 notebook；(2) 不看代码，尝试自己重写 Value 类；(3) 做 Cell 26 的三道面试题。


---

## 附录 A：时间快速参考表

| 分钟 | 动作 | Cell |
|:---|:---|:---|
| 0 | 开场，打开 notebook | 0 |
| 2 | 学习路线 + 前置知识回顾 | 1–2 |
| 5 | 跑环境准备 | 3–5 |
| 7 | 运行计算图直觉图，讲解前向/反向 | 7 |
| 10 | 逐行讲解 Value 类 | 6, 8 |
| 25 | 自动微分理论 + 测试 Micrograd | 9–11 |
| 35 | **休息 5 分钟** | — |
| 40 | 可视化计算图 + 讲解 DAG | 12–13 |
| 50 | 训练 MLP + 讲训练循环 | 14–16 |
| 60 | 理论：梯度清零/SGD/学习率 + Loss 曲线 | 17–18 |
| 67 | PyTorch 对比验证 | 19–20 |
| 75 | **休息 5 分钟** | — |
| 80 | 链式法则可视化 | 21–23 |
| 85 | 动手练习：exp + relu | 24–25 |
| 95 | 总结 + 下一章预告 | 26–28 |
| 100 | 结束 | — |


## 附录 B：关键数据快速参考

### 核心输出数值

| 来源 | 数据 | 值 |
|:---|:---|:---|
| Cell 7 直觉图 | `L = (a*b+c)*d` | `a=2,b=3,c=4,d=5 → L=50` |
| Cell 7 直觉图 | `a` 的梯度 | 15 |
| Cell 11 测试 | `d = a*b+c` | `2×(-3)+10 = 4.0` |
| Cell 11 测试 | `a.grad, b.grad, c.grad` | `-3.0, 2.0, 1.0` |
| Cell 13 神经元 | `o = tanh(n)` | `tanh(0.8814) = 0.7071` |
| Cell 13 神经元 | `w1.grad, w2.grad, b.grad` | `1.0, 0.0, 0.5` |
| Cell 15 MLP | 参数量 | 41 |
| Cell 16 训练 | Epoch 20 Loss | 0.060669 |
| Cell 16 训练 | Epoch 100 Loss | 0.007151 |
| Cell 16 训练 | 最终预测 | `[0.97, -0.97, -0.95, 0.95]` |
| Cell 16 训练 | 训练耗时 | 0.12 秒 |
| Cell 20 PyTorch | `w1.grad, w2.grad, b.grad` | `1.0, 0.0, 0.5`（与 Micrograd 一致） |
| Cell 25 exp | `exp(2)` 值和梯度 | 均为 7.3891 |

### 核心公式

- **反向传播核心**：$\frac{\partial L}{\partial u_i} \mathrel{+}= \frac{\partial L}{\partial v} \cdot \frac{\partial v}{\partial u_i}$
- **加法梯度**：直传（×1）
- **乘法梯度**：交叉相乘（∂(ab)/∂a = b）
- **tanh 梯度**：$1 - \tanh^2(x)$
- **梯度下降**：$\theta_{new} = \theta_{old} - \eta \cdot \nabla_\theta L$


## 附录 C：应急预案

### 常见问题及解决方案

| 问题 | 症状 | 解决方案 |
|:---|:---|:---|
| **字体乱码** | matplotlib 图上中文显示为方框 | 检查 Cell 5 和 Cell 7 的字体路径；使用 `SimHei` 或 `Microsoft YaHei` |
| **torch 未安装** | `ModuleNotFoundError: No module named 'torch'` | 运行 `pip install torch` 或取消 Cell 4 的注释运行 |
| **计算图图片不显示** | Cell 13 只输出文字没有图 | 检查 `%matplotlib inline` 是否生效；确认 matplotlib backend |
| **训练 Loss 不下降** | Epoch 100 Loss 仍然很大 | 随机种子问题，重启 Kernel 重新运行（权重随机初始化） |
| **练习代码报错** | Cell 25 `ValueV2` 的 exp/relu 报错 | 检查是否正确继承 Value；确认 `_backward` 闭包中用了 `+=` |
| **Cell 执行顺序错误** | `NameError: name 'Value' is not defined` | 确保按顺序运行：先 Cell 8（定义 Value），再运行后续 Cell |

### 时间不够的裁剪方案

| 剩余时间 | 裁剪策略 |
|:---|:---|
| 只剩 60 分钟 | 跳过 Cell 9 理论深入 + Cell 17 训练理论，口头简述 |
| 只剩 45 分钟 | 在上面基础上，跳过 Cell 21–23 链式法则可视化 |
| 只剩 30 分钟 | 只讲 Cell 0–11（Value 类 + 测试）+ Cell 19–20（PyTorch 验证）+ 总结 |

### 学生进度差异处理

- **快的学生**：Cell 24 有两道思考题（梯度累加、torch.no_grad()），让他们先做
- **慢的学生**：练习环节直接给出答案代码，重点理解原理而非编码
- **完全跟不上**：建议课后先看 Ch0，重点理解训练循环四步，再回来看本章
